# 6주차 랩 · 오퍼를 여러 번 만든 신청은 과정이 어떻게 다를까

**⏱ 60분** · 교재 6-1·6-2와 같은 BPI 2017 대출 신청 기록 · 개인 제출

## 🎯 이번 랩의 질문

**오퍼**는 은행이 고객에게 제시한 대출 조건이고, 한 신청에 여러 번 만들어질 수 있습니다. 첫 종료까지 오퍼를 한 번 만든 신청(`single`)과 여러 번 만든 신청(`multiple`)을 비교해, 여러 번 만든 신청에서 어느 구간이 달라지는지 봅니다.

> **여러 오퍼가 생성된 신청에서 어떤 구간을 우선 조사할까요?**

두 집단은 종료 뒤에 관측된 특성으로 나눈 것입니다. 오퍼 횟수를 바꾼 효과로 해석하지 않습니다.

교재에서 반복한 패턴을 두 집단에 다시 씁니다.

| 패턴 | 교재 | 이번 랩 |
|---|---|---|
| 묶어서 요약 — 건수·비율 | 6-1 4절 | 과제 1 |
| 조건 필터로 구간 고르기 → 횟수·신청 수 | 6-2 6절, 6-1 5절 | 과제 2·3 |
| 구간 시간의 건수·평균·중앙값 | 6-2 4절 | 과제 4 |
| 확정 신청의 미비 비율·횟수·기간 | 6-2 6절 | 과제 5·6 |
| 숫자를 근거로 판단 쓰기 | 6-2 7절 | 과제 7 |

### 진행 방법

| 표시 | 뜻 | 할 일 |
|---|---|---|
| ▶ | 제공 코드 | 그대로 실행합니다 |
| 📝 | 과제 | `None` 자리를 코드로 바꿔 채웁니다. 변수 이름은 바꾸지 않습니다 |
| ✅ | 확인 셀 | 고치지 말고 실행만 합니다 |
| 💬 | 결과 읽기 | 내 결과에서 무엇을 볼지 짚습니다. 과제 7에서 이 숫자들을 씁니다 |

- 준비 셀부터 순서대로 `Shift + Enter`로 실행합니다. 결과는 반올림하지 않고 변수에 저장합니다.
- **✅ 확인 셀의 "형식 확인 완료"는 정답이라는 뜻이 아닙니다.** 작성 여부·자료형·표 구조만 봅니다. 계산값은 제출 후 교수자가 채점하고, 과제 7의 서술은 따로 읽습니다.
- 시간: 준비 5분 · 과제 1~3 각 5분 · 과제 4 10분 · 과제 5·6 각 5분 · 과제 7 20분

## ▶ 준비

교재 6-1 4·5절(종료 시점·상태 기록)과 6-2 2·6절(구간 시간·미비 횟수)의 계산을 같은 기준으로 실행하고, 신청마다 첫 종료까지 오퍼 생성(`O_Create Offer`) 횟수를 셉니다. 다음 표가 만들어집니다.

| 표 | 한 행과 열 |
|---|---|
| `cohort` | 종료된 신청 하나. 인덱스는 신청 번호. `start`·`end`·`outcome`·`days`·`offer_count`·`offer_group`·`confirmed` |
| `edges` | 신청 안의 연속한 두 상태 구간 하나. `case_id`·`activity`·`next`·`gap`·`offer_group` |
| `accepted` | `cohort` 중 확정 신청. `rounds`는 첫 종료까지의 미비 기록 횟수 |

- `days`와 `gap`의 단위는 일이며 밤과 주말을 포함합니다. `confirmed`는 확정이면 True입니다.
- 오퍼 생성이 없는 신청이 있는지도 준비 셀에서 확인합니다(없으면 멈춤).

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. 교재 6-1·6-2와 같은 기준으로 cohort·edges·accepted 표를 만듭니다. (30초~1분)
%matplotlib inline
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

for _p in (".", "..", "../..", "../../.."):
    if os.path.exists(os.path.join(_p, "balab.py")):
        sys.path.insert(0, _p)
        break
else:
    !wget -q https://raw.githubusercontent.com/BALAB-PKNU/bizanalytics/main/balab.py

from balab import load, checker, summary, check_format

check = checker("week06")   # 자가 점검 — 정답 수치는 노트북에 없다

# [제공] 교재에서 검증한 대상·종료 시점·상태 구간을 준비한다.
log = load("loan_process")
ordered = log.reset_index().rename(columns={"index": "event_order"})
ordered = ordered.sort_values(["case_id", "timestamp", "event_order"])
starts = ordered[ordered["activity"] == "A_Create Application"].groupby("case_id")["timestamp"].min()
final = ordered[ordered["activity"].isin(["A_Pending", "A_Cancelled", "A_Denied"])]
final = final.drop_duplicates("case_id").set_index("case_id")
cohort = starts.to_frame("start")
cohort["end"] = final["timestamp"]
cohort["outcome"] = final["activity"]
cohort = cohort.dropna(subset=["end"]).copy()
cohort["days"] = (cohort["end"] - cohort["start"]).dt.total_seconds() / 86400
prior = ordered.copy()
prior["end"] = prior["case_id"].map(cohort["end"])
prior = prior[prior["end"].notna() & (prior["timestamp"] <= prior["end"])].copy()
offers = prior[prior["activity"] == "O_Create Offer"].groupby("case_id").size()
cohort["offer_count"] = offers.reindex(cohort.index).fillna(0).astype(int)
assert (cohort["offer_count"] >= 1).all(), "오퍼 생성 기록이 없는 신청의 처리 기준을 확인한다."
cohort["offer_group"] = "single"
cohort.loc[cohort["offer_count"] > 1, "offer_group"] = "multiple"
cohort["confirmed"] = cohort["outcome"] == "A_Pending"
states = prior[prior["activity"].str.startswith("A_")].copy()
states["next"] = states.groupby("case_id")["activity"].shift(-1)
states["next_ts"] = states.groupby("case_id")["timestamp"].shift(-1)
states["gap"] = (states["next_ts"] - states["timestamp"]).dt.total_seconds() / 86400
edges = states.dropna(subset=["next"]).copy()
edges["offer_group"] = edges["case_id"].map(cohort["offer_group"])
assert (edges["gap"] >= 0).all()
assert np.allclose(edges.groupby("case_id")["gap"].sum(), cohort["days"])
accepted = cohort[cohort["confirmed"]].copy()
round_counts = states[states["activity"] == "A_Incomplete"].groupby("case_id").size()
accepted["rounds"] = round_counts.reindex(accepted.index).fillna(0).astype(int)

cohort.head()

✅ `cohort`의 앞 다섯 행이 나오면 됩니다.

---
## 📝 과제 1 · 두 집단은 몇 건이고, 얼마나 확정됐나

비교에 앞서 두 집단의 규모부터 봅니다. 교재 6-1 4절에서 결과별로 묶었던 것처럼, 이번에는 `offer_group`으로 묶습니다.

- 집단별 신청 수를 `size`로 세어 `group_counts`에 담습니다.
- `confirmed`(참·거짓)의 평균, 곧 확정 비율을 `confirmed_rate`에 담습니다.

**결과 형식:** `single`·`multiple` 인덱스를 가진 길이 2의 Series 두 개

In [ ]:
# 📝 과제 1 — None 자리를 채우세요
# 힌트: cohort를 offer_group으로 묶어 한 번은 size, 한 번은 confirmed 열의 mean을 구합니다.
group_counts = None    # 집단별 신청 수
confirmed_rate = None  # 집단별 확정 비율

In [ ]:
# ✅ 확인 — 과제 1 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    if group_counts is None or confirmed_rate is None:
        check(1, group_counts=None)
    else:
        check(1, group_counts=group_counts, confirmed_rate=(confirmed_rate, 0.001))
else:
    check_format(1, group_counts=group_counts, confirmed_rate=confirmed_rate)

💬 **결과 읽기** — 두 집단의 크기는 얼마나 다른가요? 확정 비율이 다르다면, 뒤에서 확정 신청만 본 결과(과제 5·6)와 전체 종료 신청을 본 결과(과제 2~4)의 대상이 다르다는 점을 기억합니다.

---
## 📝 과제 2 · 미비 → 검증 구간은 몇 번 일어났나

6-1의 상태 지도에서 검증과 미비 사이의 반복이 큰 흐름이었습니다. 교재 6-2 6절 마지막 셀처럼 **조건 필터**로 구간을 고릅니다.

- `edges`에서 `activity`가 `A_Incomplete`이고 `next`가 `A_Validating`인 행을 `loop_edges`에 남깁니다. 두 조건은 `&`로 잇고 각각 괄호로 감쌉니다.
- `loop_edges`를 `offer_group`으로 묶어 `size`로 발생 횟수 `loop_counts`를 셉니다. 같은 신청에서 반복되면 여러 번 셉니다.

**결과 형식:** `loop_edges`는 구간 표, `loop_counts`는 길이 2의 Series

In [ ]:
# 📝 과제 2 — None 자리를 채우세요
# 힌트: 두 조건을 &로 이은 조건 필터 → offer_group으로 묶어 size. 교재 6-2 6절 마지막 셀의 필터와 같은 모양입니다.
loop_edges = None   # 미비에서 검증으로 이어진 구간 표
loop_counts = None  # 집단별 구간 발생 횟수

In [ ]:
# ✅ 확인 — 과제 2 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    check(2, loop_counts=loop_counts)
else:
    check_format(2, loop_counts=loop_counts)

💬 **결과 읽기** — 횟수는 집단 크기의 영향을 받습니다. 과제 3에서 신청 수와 신청당 횟수로 나눠 봅니다.

---
## 📝 과제 3 · 그 구간을 거친 신청 수와 신청당 횟수

구간 횟수와 그 구간을 거친 신청 수는 다릅니다(6-1 과제 3).

- `loop_edges`에서 `case_id`가 겹치는 행을 없애고(`drop_duplicates`, 교재 6-1 4절), 집단별 신청 수를 세어 `loop_cases`에 담습니다.
- `loop_counts`를 과제 1의 `group_counts`로 나눈 `loops_per_case`도 계산합니다. 분모는 그 구간을 거친 신청만이 아니라 **집단의 전체 종료 신청**입니다.

**결과 형식:** 길이 2의 Series 두 개

In [ ]:
# 📝 과제 3 — None 자리를 채우세요
# 힌트: 중복 제거 → offer_group으로 묶어 size. 신청당 횟수는 두 Series를 나누면 같은 집단끼리 맞춰 나뉩니다.
loop_cases = None      # 해당 구간을 한 번 이상 거친 신청 수
loops_per_case = None  # 집단 전체 신청당 발생 횟수

In [ ]:
# ✅ 확인 — 과제 3 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    if loop_cases is None or loops_per_case is None:
        check(3, loop_cases=None)
    else:
        check(3, loop_cases=loop_cases, loops_per_case=(loops_per_case, 0.001))
else:
    check_format(3, loop_cases=loop_cases, loops_per_case=loops_per_case)

💬 **결과 읽기** — 신청당 횟수로 보면 두 집단의 차이가 얼마나 되나요? 구간 횟수만 봤을 때와 판단이 달라지나요?

---
## 📝 과제 4 · 작성 완료에서 검증까지 걸린 시간

교재 6-2 4절에서 작성 완료 → 검증은 누적 일수가 두 번째로 큰 구간이었습니다. 이 구간의 시간이 집단별로 다른지 봅니다.

- 먼저 `edges`에서 `activity`가 `A_Complete`이고 `next`가 `A_Validating`인 행을 `phase`에 남깁니다.

In [ ]:
# 📝 과제 4 (1) — None 자리를 채우세요
# 힌트: 과제 2의 조건 필터에서 두 상태 이름만 바꿉니다.
phase = None  # 작성 완료에서 검증으로 이어진 구간 표

고른 구간의 규모와 시간을 함께 봅니다.

- `phase`를 `offer_group`으로 묶어 `size`로 `phase_counts`, `gap`의 평균으로 `phase_mean`, 중앙값으로 `phase_median`을 만듭니다. 단위는 건과 일입니다.

**결과 형식:** 길이 2의 Series 세 개

In [ ]:
# 📝 과제 4 (2) — None 자리를 채우세요
# 힌트: offer_group으로 묶은 뒤 size, gap 열의 mean, gap 열의 median을 각각 구합니다.
phase_counts = None  # 집단별 구간 수
phase_mean = None    # 구간 시간 평균, 일
phase_median = None  # 구간 시간 중앙값, 일

In [ ]:
# ✅ 확인 — 과제 4 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    if phase_counts is None or phase_mean is None or phase_median is None:
        check(4, phase_counts=None)
    else:
        check(4, phase_counts=phase_counts, phase_mean=(phase_mean, 0.01), phase_median=(phase_median, 0.01))
else:
    check_format(4, phase_counts=phase_counts, phase_mean=phase_mean, phase_median=phase_median)

💬 **결과 읽기** — 평균과 중앙값이 모두 같은 방향으로 다른가요? 평균만 크게 다르다면 일부 긴 구간이 평균을 끌어올렸을 수 있습니다.

---
## 📝 과제 5 · 확정 신청에서 미비는 얼마나 널리 생겼나

교재 6-2 6절의 계산을 두 집단에 적용합니다. 이번에는 확정 신청(`accepted`)만 봅니다.

- 집단별 확정 신청 수를 `accepted_counts`에 담습니다.
- `rounds`가 0보다 크면 True인 열 `has_incomplete`를 만들고, 집단별 평균을 `incomplete_rate`에 담습니다.

**결과 형식:** 길이 2의 Series 두 개. 과제 1의 전체 종료 신청과 분모가 다릅니다.

In [ ]:
# 📝 과제 5 — None 자리를 채우세요
# 힌트: 교재 6-2 6절의 '0보다 큰지 비교 → 평균'을 새 열로 만든 뒤 offer_group으로 묶어 평균을 구합니다.
accepted_counts = None  # 집단별 확정 신청 수
incomplete_rate = None  # 확정 신청 중 미비가 있는 비율

In [ ]:
# ✅ 확인 — 과제 5 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    if accepted_counts is None or incomplete_rate is None:
        check(5, accepted_counts=None)
    else:
        check(5, accepted_counts=accepted_counts, incomplete_rate=(incomplete_rate, 0.001))
else:
    check_format(5, accepted_counts=accepted_counts, incomplete_rate=incomplete_rate)

---
## 📝 과제 6 · 확정 신청의 미비 횟수와 전체 기간

교재 6-2 6절 빈칸 셀처럼 묶어서 요약하되, 이번에는 `offer_group`으로 묶습니다.

- `rounds`의 평균 `mean_rounds`와 `days`의 평균 `mean_days`를 구합니다. 미비가 없는 신청의 0회도 포함합니다.

**결과 형식:** 길이 2의 Series 두 개

In [ ]:
# 📝 과제 6 — None 자리를 채우세요
# 힌트: accepted를 offer_group으로 묶어 rounds 열과 days 열의 mean을 각각 구합니다.
mean_rounds = None  # 확정 신청의 집단별 평균 미비 횟수
mean_days = None    # 확정 신청의 집단별 평균 전체 일수

In [ ]:
# ✅ 확인 — 과제 6 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    check(6, mean_rounds=(mean_rounds, 0.01), mean_days=(mean_days, 0.01))
else:
    check_format(6, mean_rounds=mean_rounds, mean_days=mean_days)

💬 **결과 읽기** — 과제 4의 구간 시간(작성 완료 → 검증)과 여기서 잰 생성부터 확정까지의 전체 기간을 구별합니다. 미비 횟수와 전체 기간이 같은 방향으로 다른지 봅니다.

---
## 📝 과제 7 · 본인의 판단과 설명

계산 결과를 근거로 본인의 판단을 씁니다. 사용한 수치를 함께 적습니다.

1. 조사할 구간 하나를 고르고, 발생 규모(횟수·신청 수·신청당 횟수)와 시간을 수치로 설명합니다. 운영 담당자에게 확인할 사유나 기록도 제시합니다.
2. 두 집단의 차이를 근거로 "오퍼를 한 번만 만들면 빨라진다"고 말할 수 있는지 판단합니다. 확정 신청만 본 결과의 적용 대상과, 오퍼 생성 횟수가 끝난 뒤 정해진 특성이라는 점을 반영합니다.

자동 채점은 하지 않습니다.

✏️ **나의 답:** (여기를 두 번 눌러 적습니다)

---
## ✅ 제출 전 확인

아래 셀을 실행해 과제 1~6의 작성 여부와 형식을 확인합니다. "형식 확인 완료"는 계산값이 맞다는 뜻이 아닙니다. 결과를 직접 읽고 과제 7을 작성해 제출합니다. 실제 값 채점은 제출 후 진행합니다.

In [ ]:
# ✅ 제출 전 확인 — 과제 1~6의 작성 여부와 형식
summary()